In [32]:
import numpy as np
import scipy as sp
import pyscf
from pyscf import gto, scf

mol = gto.Mole()
mol.atom = """
H    0.0    0.0    0.0
H    0.74    0.0    0.0
"""
mol.basis = '6-31g'
mol.charge = 0
mol.spin = 0
mol.build()

mf = scf.UHF(mol)

# Tighten SCF convergence criteria
mf.conv_tol = 1e-12       # Energy convergence tolerance
mf.conv_tol_grad = 1e-12   # Gradient convergence tolerance
mf.max_cycle = 1000        # Max SCF iterations
mf.kernel()  

converged SCF energy = -1.12675531719697  <S^2> = 2.220446e-16  2S+1 = 1


-1.126755317196973

In [33]:
mf.energy_nuc()

0.7151043390810812

In [34]:
print(mf.mol.nao)

4


In [35]:
from scipy.sparse.csgraph import reverse_cuthill_mckee

def make_so(mf):
    alpha_mo = mf.mo_coeff[0]
    beta_mo = mf.mo_coeff[1]

    h1e_ao = mf.get_hcore()

    h2e_ao = mf.mol.intor('int2e', aosym = "s1")

    nao = mf.mol.nao
    nso = 2*nao

    h1e_alpha = np.einsum("ij,ia,jb->ab", h1e_ao, alpha_mo, alpha_mo, optimize=True)
    h1e_beta = np.einsum('ij,ia,jb->ab', h1e_ao, beta_mo, beta_mo, optimize=True)

    h2e_aa = np.einsum('ijkl,ia,jb,kc,ld->abcd', h2e_ao, alpha_mo, alpha_mo, alpha_mo, alpha_mo, optimize=True)
    h2e_bb = np.einsum('ijkl,ia,jb,kc,ld->abcd', h2e_ao, beta_mo, beta_mo, beta_mo, beta_mo, optimize=True)
    h2e_ab = np.einsum('ijkl,ia,jb,kc,ld->abcd', h2e_ao, alpha_mo, alpha_mo, beta_mo, beta_mo, optimize=True)
    h2e_ba = np.einsum('ijkl,ia,jb,kc,ld->abcd', h2e_ao, beta_mo, beta_mo, alpha_mo, alpha_mo, optimize=True)

    h2e_so = np.zeros((nso,nso,nso,nso))
    h2e_so[:nao,:nao,:nao,:nao] = h2e_aa
    h2e_so[nao:,nao:,nao:,nao:] = h2e_bb
    h2e_so[nao:,nao:,:nao,:nao] = h2e_ba
    h2e_so[:nao,:nao,nao:,nao:] = h2e_ab


    h1e_so = np.zeros((nso,nso))
    h1e_so[:nao,:nao] = h1e_alpha
    h1e_so[nao:,nao:] = h1e_beta

    mo_coeff = np.zeros((nso,nso))
    mo_coeff[:nao,:nao] = alpha_mo
    mo_coeff[nao:,nao:] = beta_mo

    return h1e_so, h2e_so, mo_coeff


def orbital_ordering(mf):
    h1e_so, h2e_so, mo_coeff = make_so(mf)

    # now we want to reorder h1e_so indices

    h1e_pattern = np.abs(h1e_so) > 1e-10
    h1e_graph = sp.sparse.csr_matrix(h1e_pattern.astype(int))
    
    perm = reverse_cuthill_mckee(h1e_graph)

    h1e_so_reordered = h1e_so[np.ix_(perm,perm)]
    h2e_so_reordered = h2e_so[np.ix_(perm,perm,perm,perm)]
    mo_coeff_reordered = mo_coeff[:,perm]

    h2e_so_reordered = 0.5 * h2e_so_reordered # NEWLINE TO ACCOUNT FOR THE 1/2 FACTOR THAT WASN'T PREVIOUSLY PRESENT

    return h1e_so_reordered, h2e_so_reordered, mo_coeff_reordered, perm


In [6]:
from pyscf import fci

cisolver = pyscf.fci.FCI(mf)
cisolver.kernel()

(-2.251602269008274,
 FCIvector([[ 9.87938249e-01, -3.33066907e-16, -1.12793470e-03,
              1.13468937e-03, -2.93779168e-16,  1.37800037e-02],
            [-2.88376445e-16,  8.05291031e-02,  1.59004334e-16,
              9.42879256e-18,  4.22789565e-02, -4.62557629e-17],
            [-1.12793470e-03,  2.53265802e-17, -6.07056513e-02,
             -5.60589602e-02, -1.22807829e-17,  1.10988886e-03],
            [-1.13468937e-03, -4.15051541e-17,  5.60589602e-02,
              4.48032518e-02, -1.15440371e-17,  2.08544062e-03],
            [ 2.70584523e-16, -4.22789565e-02,  3.50615121e-17,
              1.52543856e-16, -3.77162600e-02,  2.70453472e-17],
            [-1.37800037e-02,  4.35672972e-17, -1.10988886e-03,
              2.08544062e-03,  3.66604807e-17, -1.10346129e-02]]))

In [29]:
def exact_diagonalization(mf):
    t, g, C, p = orbital_ordering(mf)
    nso = t.shape[0]

    Z = sp.sparse.csr_matrix([[1, 0], [0, -1]])
    I = sp.sparse.identity(2, format='csr')
    a = sp.sparse.csr_matrix([[0, 0], [1, 0]])  # annihilation
    a_dag = sp.sparse.csr_matrix([[0, 1], [0, 0]])  # creation

    lowering_ops = []
    raising_ops = []
    
    for i in range(nso):
        ops_lower = [Z if j < i else I for j in range(nso)]
        ops_raise = [Z if j < i else I for j in range(nso)]
        ops_lower[i] = a
        ops_raise[i] = a_dag

        a_i = ops_lower[0]
        a_i_dag = ops_raise[0]
        for op in ops_lower[1:]:
            a_i = sp.sparse.kron(a_i, op)
        for op in ops_raise[1:]:
            a_i_dag = sp.sparse.kron(a_i_dag, op)
            
        lowering_ops.append(a_i)
        raising_ops.append(a_i_dag)

    H = sp.sparse.csr_matrix((2**nso, 2**nso))

    for i in range(nso):
        for j in range(nso):
            H += t[i,j] * (raising_ops[i] @ lowering_ops[j])

    for i in range(nso):
        for j in range(nso):
            for k in range(nso):
                for l in range(nso):
                    term = raising_ops[i] @ raising_ops[j] @ lowering_ops[k] @ lowering_ops[l]
                    # H += 0.5 * g[i,l,j,k] * term  # Note g[i,l,j,k] matches (il|jk) # ADDED 0.5 TO ORBITALREORDERING, SO CHANGING THIS
                    H += g[i,l,j,k] * term

    
    print(np.shape(H))


    eigval, eigvec = sp.sparse.linalg.eigsh(H, k=1, which='SA')
    eigval = eigval + mf.energy_nuc()
    return eigval[0]

In [40]:
exact_diagonalization(mf)

KeyboardInterrupt: 

In [ ]:
# def exact_diagonalization(mf):
#     # t, g, C, perm = orbital_ordering(mf)

#     t, g, C = make_so(mf)

#     nso = 2*mf.mol.nao

#     # operators as sparse matrices
#     Z = sp.sparse.csr_matrix([[1, 0], [0, -1]])
#     I = sp.sparse.identity(2, format='csr')
#     a = sp.sparse.csr_matrix([[0, 0], [1, 0]])     # annihilation
#     a_t = sp.sparse.csr_matrix([[0, 1], [0, 0]])   # creation

#     lowering_operators = []
#     raising_operators = []

#     for i in range(nso):
#         ops_lower = []
#         ops_raise = []
#         for j in range(nso):
#             if j < i:
#                 ops_lower.append(Z)
#                 ops_raise.append(Z)
#             elif j == i:
#                 ops_lower.append(a)
#                 ops_raise.append(a_t)
#             else:
#                 ops_lower.append(I)
#                 ops_raise.append(I)
        
#         # Tensor product: left to right
#         a_i = ops_lower[0]
#         a_i_dag = ops_raise[0]
#         for op in ops_lower[1:]:
#             a_i = sp.sparse.kron(a_i, op, format='csr')
#         for op in ops_raise[1:]:
#             a_i_dag = sp.sparse.kron(a_i_dag, op, format='csr')
        
#         lowering_operators.append(a_i)
#         raising_operators.append(a_i_dag)
    
#     H = sp.sparse.csr_matrix((2**nso,2**nso))
#     print(np.shape(H))

#     two_body_ops = {}
#     for i in range(nso):
#         for j in range(nso):
#             for k in range(nso):
#                 for l in range(nso):
#                     op = raising_operators[i] @ raising_operators[j] @ lowering_operators[k] @ lowering_operators[l]
#                     two_body_ops[(i,j,k,l)] = op


#     for i in range(nso):
#         for j in range(nso):
#             H += t[i,j] * raising_operators[i] @ lowering_operators[j]
#             for k in range(nso):
#                 for l in range(nso):
#                     H += 0.5* g[i,l,j,k] * two_body_ops[(i,j,k,l)]
    
#     eigval, eigvec = sp.sparse.linalg.eigsh(H, 1, which = 'SA')

#     return eigval
            
        

    

In [19]:
import matplotlib.pyplot as plt

# plt.matshow(h1e_so)

In [ ]:
# h1e_so_reordered, h2e_so_reordered, mo_coeff_reordered, perm = orbital_ordering(mf)

In [ ]:
# def lattice_construction_dmrg(mf):
#     # begin with leftmost and then rightmost double blcoks

    
#     t, g, C, _ = orbital_ordering(mf)

#     w = g - np.transpose(g, (2, 1, 0, 3))

#     x = g - np.transpose(g, (2, 1, 0, 3)) - np.transpose(g, (0, 3, 2, 1)) + np.transpose(g, (2, 3, 0, 1))

#     nso = mf.mol.nao * 2

#     left_O = []
#     right_O = []

#     # now construct left block, beginning with 2 sites.

#     H_l = np.zeros((4,4))
#     H_r = np.zeros((4,4))

#     I = np.eye(2)
#     Z = [[1, 0],
#          [0, -1]]

#     a = [[0, 0], 
#          [1, 0]]
    
#     a_t = [[0, 1],
#            [0, 0]]
    
#     a = np.array(a)
#     a_t = np.array(a_t)
#     Z = np.array(Z)

#     lower_L = []
#     lower_R = []
#     raise_L = []
#     raise_R = []

#     lower_L.append(np.kron(a,I))
#     raise_L.append(np.kron(a_t, I))
#     lower_L.append(np.kron(Z,a))
#     raise_L.append(np.kron(Z,a_t))


#     lower_R.append(np.kron(Z,a))
#     raise_R.append(np.kron(Z,a_t))
#     lower_R.append(np.kron(a,I))
#     raise_R.append(np.kron(a_t, I))

#     Lzstring = np.kron(Z, Z)
#     LIstring = np.kron(I, I)
#     RIstring = np.kron(I, I)

#     for i in range(2):
#         for j in range(2):
#             H_l += t[i,j] * raise_L[i] @ lower_L[j] # a_dag a
#             H_r += t[nso-i-1,nso-j-1] * raise_R[i] @ lower_R[j] # a_dag a
            
#             for k in range(2):
#                 for l in range(2):
#                     H_l += g[i,l,j,k] * raise_L[i] @ raise_L[j] @ lower_L[k] @ lower_L[l]
#                     H_r += g[nso-i-1, nso-l-1, nso-j-1, nso-k-1] * raise_R[i] @ raise_R[j] @ lower_R[k] @ lower_R[l]


#     def make_superblock_and_diagonalize(H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R):
#         size_L = np.shape(H_l)[0]
#         size_R = np.shape(H_r)[0]

#         # H_noninteracting = np.kron(H_l, np.eye(size_R)) + np.kron(np.eye(size_L), H_r)
#         H_noninteracting  = np.kron(H_l, RIstring) + np.kron(LIstring, H_r)

#         H_interacting = np.zeros_like(H_noninteracting)
#         length_L = np.shape(lower_L)[0]
#         length_R = np.shape(lower_R)[0]

#         print(np.shape(lower_L))
#         print(np.shape(lower_R))

#         for i in range(length_R):
#             for j in range(length_L):
#                 H_interacting += t[nso-i-1,j] * (np.kron(Lzstring, raise_R[i]) @ np.kron(lower_L[j], RIstring) + np.kron(raise_L[j], RIstring) @ np.kron(Lzstring, lower_R[i]))
#                 for k in range(length_L):
#                     for l in range(length_L):
#                         H_interacting += (g[nso-i-1,l,j,k] - g[j,l,nso-i-1,k]) * np.kron(Lzstring, raise_R[i]) @ np.kron(raise_L[j], RIstring) @ np.kron(lower_L[k], RIstring) @ np.kron(lower_L[l], RIstring)
        
#         for i in range(length_L):
#             for j in range(length_R):
#                 for k in range(length_R):
#                     for l in range(length_R):
#                         H_interacting += (g[i, nso-l-1, nso-j-1, nso-k-1] - g[nso-j-1, nso-l-1, i, nso-k-1]) * np.kron(raise_L[i], RIstring) @ np.kron(Lzstring, raise_R[j]) @ np.kron(Lzstring, lower_R[k]) @ np.kron(Lzstring, lower_R[l])

#         for i in range(length_R):
#             for j in range(length_R):
#                 for k in range(length_L):
#                     for l in range(length_L):
#                         H_interacting += (g[nso-i-1, l, nso-j-1, k]) * (np.kron(Lzstring, raise_R[i]) @ np.kron(Lzstring, raise_R[j]) @ np.kron(lower_L[k], RIstring) @ np.kron(lower_L[l], RIstring) + np.kron(raise_L[k], RIstring) @ np.kron(raise_L[l], RIstring) @ np.kron(Lzstring, lower_R[i]) @ np.kron(Lzstring, lower_R[j]))
        
#         for i in range(length_R):
#             for k in range(length_R):
#                 for j in range(length_L):
#                     for l in range(length_L):
#                         H_interacting += x[nso-i-1, l, j, nso-k-1] * np.kron(Lzstring, raise_R[i]) @ np.kron(raise_L[j], RIstring) @ np.kron(Lzstring, lower_R[k]) @ np.kron(lower_L[l], RIstring)

#         H_superblock = H_noninteracting + H_interacting

#         H_superblock_sparse = sp.sparse.csr_matrix(H_superblock)

#         E0, Psi = sp.sparse.linalg.eigsh(H_superblock_sparse, 1) # Psi has dimensions 2**lengthL * 2**lengthR

#         print(np.shape(Psi))

#         Psi_reshaped = np.reshape(Psi, (2**length_L, 2**length_R))

#         dmL = np.einsum('ij,kj->ik', Psi_reshaped, Psi_reshaped, optimize = True)
#         dmR = np.einsum('ij,ik->jk', Psi_reshaped, Psi_reshaped, optimize = True)

#         return Psi_reshaped, E0, dmL, dmR
    

#     def getNewBasis(dmL, dmR, tol = 0.0):

#         schmidt_L, basis_L = np.linalg.eigh(-dmL)
#         schmidt_R, basis_R = np.linalg.eigh(-dmR)

#         schmidt_L = -schmidt_L
#         schmidt_R = -schmidt_R

#         if np.shape(schmidt_L)[0] > 6 and np.shape(schmidt_R)[0] > 6:
#             mask_L = np.abs(schmidt_L) > tol
#             mask_R = np.abs(schmidt_R) > tol

#             print("original shape: ", np.shape(basis_L), np.shape(basis_R))
#             basis_L = basis_L[:, mask_L]
#             basis_R = basis_R[:, mask_R]
#             print("final shape: ", np.shape(basis_L), np.shape(basis_R))


#         return basis_L, basis_R
    
#     def rotate_everything(H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R, basis_L, basis_R):
#         H_l_rot = np.einsum('ij,ia,jb->ab', H_l, basis_L, basis_L)
#         H_r_rot = np.einsum('ij,ia,jb->ab', H_r, basis_R, basis_R)

#         Lzstring_rot = np.einsum('ij,ia,jb->ab', Lzstring, basis_L,basis_L)
#         LIstring_rot = np.einsum('ij,ia,jb->ab', LIstring, basis_L, basis_L)

#         RIstring_rot = np.einsum('ij,ia,jb->ab', RIstring, basis_R, basis_R)

#         lower_L_rot = [np.einsum('ij,ia,jb->ab', op, basis_L, basis_L) for op in lower_L]
#         raise_L_rot = [np.einsum('ij,ia,jb->ab', op, basis_L, basis_L) for op in raise_L]

#         lower_R_rot = [np.einsum('ij,ia,jb->ab', op, basis_R, basis_R) for op in lower_R]
#         raise_R_rot = [np.einsum('ij,ia,jb->ab', op, basis_R, basis_R) for op in raise_R]

#         return H_l_rot, H_r_rot, Lzstring_rot, LIstring_rot, RIstring_rot, lower_L_rot, lower_R_rot, raise_L_rot, raise_R_rot
    

#     def extend_Left(H_l, Lzstring, LIstring, lower_L, raise_L):
#         # these existing operators have already been rotated.

#         new_a = np.kron(Lzstring, a)
#         new_a_dag = np.kron(Lzstring, a_t)

#         mod_Lzstring = np.kron(Lzstring, Z)

#         mod_LIstring = np.kron(LIstring, I)

#         mod_lower_L = [np.kron(a_i, I) for a_i in lower_L]
#         mod_raise_L = [np.kron(a_idag, I) for a_idag in raise_L]

#         # now i need to also extend my Hamiltonian.
#         # suppose we only add a single site; so the H_Bl hamiltonian has only a kinetic term associated w it.

#         # we need the index of the newly added left site and newly added right site. to do this, use the length
#         # of the lower_L and raise_L strings.

#         idx = np.shape(lower_L)[0]  # because we index lower_L starting at zero but this returns the full length, the
#                                     # index being returned is actually the next index

#         H_noninteracting = np.kron(H_l, I) + np.kron(LIstring, t[idx, idx] * a_t @ a)

#         H_interacting = np.zeros_like(H_noninteracting)

#         for j in range(idx):
#             H_interacting += t[idx, j] * (new_a_dag @ mod_lower_L[j] + mod_raise_L[j] @ new_a)

#             # H_interacting += g[j, idx, idx, idx] * (mod_raise_L[j] @ new_a_dag @ new_a @ new_a)

#         #     for k in range(idx):
#         #         for l in range(idx):
#         #             H_interacting += (g[idx, l, j, k] -
#         #                                g[j, l, idx, k]) * (new_a_dag @ mod_raise_L[j] @ mod_lower_L[k] @ mod_lower_L[l])
                    
#         # for k in range(idx):
#         #     for l in range(idx):

#         #         H_interacting += g[idx, l, idx, k] * (new_a_dag @ new_a_dag @ mod_lower_L[k] @ mod_lower_L[l] 
#         #                              + mod_raise_L[k] @ mod_raise_L[l] @ new_a @ new_a)

#         for j in range(idx):
#             for k in range(idx):
#                 for l in range(idx):

#                     H_interacting += w[idx, l, j, k] * (new_a_dag @ mod_raise_L[j] @ mod_lower_L[k] @ mod_lower_L[l])

#         for i in range(idx):
#             H_interacting += w[i, idx, idx, idx] * (mod_raise_L[i] @ new_a_dag @ new_a @ new_a)

#         for k in range(idx):
#             for l in range(idx):
#                 H_interacting += g[idx, l, idx, k] * (new_a_dag @ new_a_dag @ mod_lower_L[k] @ mod_lower_L[l] 
#                                                       + mod_raise_L[k] @ mod_raise_L[l] @ new_a @ new_a )

#         for j in range(idx):
#             for l in range(idx):

#                 H_interacting += x[idx, l, j, idx] * new_a_dag @ mod_raise_L[j] @ new_a @ mod_lower_L[l]
            
    
#         new_H_L = H_noninteracting + H_interacting

#         mod_raise_L.append(new_a_dag)
#         mod_lower_L.append(new_a)

#         return new_H_L, mod_Lzstring, mod_LIstring, mod_lower_L, mod_raise_L
    

#     def extend_Right(H_r, RIstring, lower_R, raise_R):

#         new_a = np.kron(a, RIstring)
#         new_a_dag = np.kron(a_t, RIstring)

#         mod_RIstring = np.kron(I, RIstring)

#         mod_raise_R = [np.kron(Z, ai_dag) for ai_dag in raise_R]
#         mod_lower_R = [np.kron(Z, ai) for ai in lower_R]

#         idx = np.shape(lower_R)[0]

#         H_noninteracting = np.kron(I, H_r) + np.kron(t[nso-1-idx, nso-1-idx] * a_t @ a, RIstring)

#         H_interacting = np.zeros_like(H_noninteracting)

#         for j in range(idx):
#             H_interacting += t[nso-1-idx, nso-1-j] * (new_a_dag @ mod_lower_R[j] + mod_raise_R[j] @ new_a)

#         for j in range(idx):
#             for k in range(idx):
#                 for l in range(idx):

#                     H_interacting += w[nso-1-idx, nso-1-l, nso-1-j, nso-1-k] * (new_a_dag @ mod_raise_R[j] @ mod_lower_R[k] @ mod_lower_R[l])

#         for i in range(idx):
#             H_interacting += w[nso-1-i, nso-1-idx, nso-1-idx, nso-1-idx] * (mod_raise_R[i] @ new_a_dag @ new_a @ new_a)

#         for k in range(idx):
#             for l in range(idx):
#                 H_interacting += g[nso-1-idx, nso-1-l, nso-1-idx, nso-1-k] * (new_a_dag @ new_a_dag @ mod_lower_R[k] @ mod_lower_R[l] 
#                                                                               + mod_raise_R[k] @ mod_raise_R[l] @ new_a @ new_a)
        
#         for j in range(idx):
#             for l in range(idx):
#                 H_interacting += x[nso-1-idx, nso-1-l, nso-1-j, nso-1-idx] * new_a_dag @ mod_raise_R[j] @ new_a @ mod_lower_R[l]

#         new_H_R = H_noninteracting+H_interacting

#         mod_raise_R.append(new_a_dag)
#         mod_lower_R.append(new_a)

#         return new_H_R, mod_RIstring, mod_lower_R, mod_raise_R
    

#     # DOUBLE CHECK EVERYTHING ABOVE

#     # begin with two on left, two on right
#     # superblock, diagonalize, truncate
#     # extend left
#     # extend right
#     # superblock, diagonalize, truncate
#     # how many times do we need to superblock/diagonalize/truncate?
#     # (nso-4)/2?

#     # suppose i had 10;
#     # (2,2) --> block-decimate --> (2',2') --> (3,3)
#     # (3,3) --> block-decimate --> (3',3') --> (4,4)
#     # (4,4) --> block-decimate --> (4',4') --> (5,5)
#     # (5,5) --> block-decimate --> (5',5'), E0

#     # need to extend left and extend right a total of (nso-4)/2 times
#     # need to block-decimate (nso-4)/2 + 1 times; the final time being the one where we get E0

#     niterations = (nso-4)//2
#     print(nso)
#     print(niterations)

#     for iteration_number in range(niterations+1):
#         print("H_l, H_r shapes:", np.shape(H_l), np.shape(H_r))
#         Psi_reshaped, E0, dmL, dmR = make_superblock_and_diagonalize(H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R)
#         print(E0)
#         basis_L, basis_R = getNewBasis(dmL, dmR)
#         H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R = rotate_everything(H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R, basis_L, basis_R)
#         H_l, Lzstring, LIstring, lower_L, raise_L = extend_Left(H_l, Lzstring, LIstring, lower_L, raise_L)
#         H_r, RIstring, lower_R, raise_R = extend_Right(H_r, RIstring, lower_R, raise_R)
    
#     Psi_reshaped, E0, dmL, dmR = make_superblock_and_diagonalize(H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R)
 
#     print(E0 + mf.energy_nuc())

#     return E0


In [41]:
import scipy as sp
import numpy as np

def lattice_construction_dmrg_retry(mf):
    # begin with leftmost and then rightmost double blcoks

    
    t, g, C, _ = orbital_ordering(mf)

    print("t shape:", np.shape(t))

    # g_iljk --> v_ijkl
    v = np.transpose(g, (0, 2, 3, 1))

    # w = g - np.transpose(g, (2, 1, 0, 3))
    w = v - np.transpose(v, (1, 0, 2, 3)) # w_ijkl

    # x = g - np.transpose(g, (2, 1, 0, 3)) - np.transpose(g, (0, 3, 2, 1)) + np.transpose(g, (2, 3, 0, 1))
    x = v - np.transpose(v, (1, 0, 2, 3)) - np.transpose(v, (0, 1, 3, 2)) + np.transpose(v, (1, 0, 3, 2))
    # now for all 2-el terms use ijkl and DO NOT USE g use v instead.


    nso = mf.mol.nao * 2

    print("nso:", nso)

    left_O = []
    right_O = []

    # now construct left block, beginning with 2 sites.

    H_l = np.zeros((4,4))
    H_r = np.zeros((4,4))

    I = np.eye(2)
    Z = [[1, 0],
         [0, -1]]

    a = [[0, 0], 
         [1, 0]]
    
    a_t = [[0, 1],
           [0, 0]]
    
    LIstring = np.kron(I, I)
    LZstring = np.kron(Z, Z)
    RIstring = np.kron(I, I)

    raise_L = []
    raise_R = []
    lower_L = []
    lower_R = []


    raise_L.append(np.kron(a_t, I))
    raise_L.append(np.kron(Z, a_t))
    lower_L.append(np.kron(a,I))
    lower_L.append(np.kron(Z, a))

    raise_R.append(np.kron(Z, a_t))
    raise_R.append(np.kron(a_t, I))
    lower_R.append(np.kron(Z, a))
    lower_R.append(np.kron(a, I))

    H_L = np.zeros((4,4))
    H_R = np.zeros((4,4))
    
    for i in range(2):
        for j in range(2):
            H_L += t[i,j] * raise_L[i] @ lower_L[j]
            for k in range(2):
                for l in range(2):
                    H_L += v[i,j,k,l] * raise_L[i] @ raise_L[j] @ lower_L[k] @ lower_L[l]

    for i in range(2):
        for j in range(2):
            H_R += t[nso-1-i, nso-1-j] * raise_R[i] @ lower_R[j]
            for k in range(2):
                for l in range(2):
                    H_R += v[nso-1-i, nso-1-j, nso-1-k, nso-1-l] * raise_R[i] @ raise_R[j] @ lower_R[k] @ lower_R[l]

    print("Initial left and right Hamiltonians constructed and operators added to the list.")
    print("Current lengths:")
    print("Left:", len(lower_L))
    print("Right:", len(lower_R))


    def superblock():

        nonlocal H_L, H_R, LIstring, LZstring, RIstring, raise_L, raise_R, lower_L, lower_R

        H_noninteracting = np.kron(H_L, RIstring) + np.kron(LIstring, H_R)
        H_interacting = np.zeros_like(H_noninteracting)

        left_sites = len(raise_L)
        right_sites = len(raise_R)

        temp_raise_L = [np.kron(op, RIstring) for op in raise_L]
        temp_lower_L = [np.kron(op, RIstring) for op in lower_L]
        temp_raise_R = [np.kron(LZstring, op) for op in raise_R]
        temp_lower_R = [np.kron(LZstring, op) for op in lower_R]


        for i in range(right_sites):
            for j in range(left_sites):
                H_interacting += t[nso-i-1, j] * (temp_raise_R[i] @ temp_lower_L[j] + temp_raise_L[j] @ temp_lower_R[i])


        for i in range(right_sites):
            for j in range(left_sites):
                for k in range(left_sites):
                    for l in range(left_sites):
                        H_interacting += w[nso-i-1, j, k, l] * (
                            temp_raise_R[i] @ temp_raise_L[j] @ temp_lower_L[k] @ temp_lower_L[l]
                        )

        for i in range(left_sites):
            for j in range(right_sites):
                for k in range(right_sites):
                    for l in range(right_sites):

                        H_interacting += w[i, nso-1-j, nso-1-k, nso-1-l] * (
                            temp_raise_L[i] @ temp_raise_R[j] @ temp_lower_R[k] @ temp_lower_R[l]
                        )

        for i in range(right_sites):
            for j in range(right_sites):
                for k in range(left_sites):
                    for l in range(left_sites):

                        H_interacting += v[nso-1-i, nso-1-j, k, l] * (
                            temp_raise_R[i] @ temp_raise_R[j] @ temp_lower_L[k] @ temp_lower_L[l] 
                            + temp_raise_L[k] @ temp_raise_L[l] @ temp_lower_R[i] @ temp_lower_R[j]
                        )

        for i in range(right_sites):
            for k in range(right_sites):
                for j in range(left_sites):
                    for l in range(left_sites):

                        H_interacting += x[nso-i-1, j, nso-k-1, l] * temp_raise_R[i] @ temp_raise_L[j] @ temp_lower_R[k] @ temp_lower_L[l]

        H_total = H_noninteracting + H_interacting

        # H_total_sparse = sp.sparse.csr_matrix(H_total)
        # E0, Psi = sp.sparse.linalg.eigsh(H_total_sparse, k=1, which='SA')

        E0, Psi = np.linalg.eigh(H_total)

        print("Energy: ", E0[0])

        return E0[0], Psi[:,0]

    def decimate(psi, tol=1e-10):

        nonlocal H_L, H_R, raise_L, raise_R, lower_L, lower_R, LIstring, LZstring, RIstring

        leftshape = np.shape(H_L)[0]
        rightshape = np.shape(H_R)[0]

        print("Left shape:", leftshape)
        print("Right shape:", rightshape)

        psi_reshape = np.reshape(psi, (leftshape, rightshape))

        dm_L = np.einsum('ij,kj->ik', psi_reshape, psi_reshape, optimize = True)
        dm_R = np.einsum('ij,ik->jk', psi_reshape, psi_reshape, optimize = True)


        # schmidt_L, basis_L = np.linalg.eigh(-dm_L)
        # schmidt_R, basis_R = np.linalg.eigh(-dm_R)

        schmidt_L, basis_L = np.linalg.eigh(dm_L)
        schmidt_R, basis_R = np.linalg.eigh(dm_R)

        schmidt_L = schmidt_L[::-1]
        schmidt_R = schmidt_R[::-1]
        basis_L = basis_L[:,::-1]
        basis_R = basis_R[:,::-1]

        # schmidt_L = -schmidt_L
        # schmidt_R = -schmidt_R

        if leftshape > 7 and rightshape > 7:
        #     # mask_L = np.abs(schmidt_L) >= tol
        #     # mask_R = np.abs(schmidt_R) >= tol
        #     # print("Old shape (left): ", np.shape(basis_L))
        #     # basis_L = basis_L[:, mask_L]
        #     # print("New shape: ", np.shape(basis_L))
        #     # basis_R = basis_R[:, mask_R]

        #     # maybe instead of this, we keep the five lowest vectors?
            basis_L = basis_L[:,:7]
            basis_R = basis_R[:,:7]



        # now we transform the operators
        H_L = np.einsum('ij,ia,jb->ab', H_L, basis_L, basis_L, optimize = True)
        H_R = np.einsum('ij,ia,jb->ab', H_R, basis_R, basis_R, optimize = True)

        raise_L = [np.einsum('ij,ia,jb->ab', op, basis_L, basis_L, optimize = True) for op in raise_L]
        lower_L = [np.einsum('ij,ia,jb->ab', op, basis_L, basis_L, optimize = True) for op in lower_L]

        raise_R = [np.einsum('ij,ia,jb->ab', op, basis_R, basis_R, optimize = True) for op in raise_R]
        lower_R = [np.einsum('ij,ia,jb->ab', op, basis_R, basis_R, optimize = True) for op in lower_R]

        LIstring = np.einsum('ij,ia,jb->ab', LIstring, basis_L, basis_L, optimize = True)
        RIstring = np.einsum('ij,ia,jb->ab', RIstring, basis_R, basis_R, optimize = True)
        LZstring = np.einsum('ij,ia,jb->ab', LZstring, basis_L, basis_L, optimize = True)


    ### when extending, Garnet adds two sites at a time -- I add only one. IDK if this makes a difference?
    def extend_left():

        nonlocal LZstring, LIstring, LZstring, a, a_t, raise_L, lower_L, t, H_L, I, Z

        new_raise_L = np.kron(LZstring, a_t)
        new_lower_L = np.kron(LZstring, a)

        idx = len(lower_L)

        lower_L = [np.kron(op, I) for op in lower_L]
        raise_L = [np.kron(op, I) for op in raise_L]

        lower_L.append(new_lower_L)
        raise_L.append(new_raise_L)

        LIstring = np.kron(LIstring, I)
        LZstring = np.kron(LZstring, Z)

        H_noninteracting = np.kron(H_L, I) + t[idx,idx] * new_raise_L @ new_lower_L
        H_interacting = np.zeros_like(H_noninteracting)

        for j in range(idx):
            H_interacting += t[idx, j] * (raise_L[idx] @ lower_L[j] + raise_L[j] @ lower_L[idx])

        for j in range(idx):
            for k in range(idx):
                for l in range(idx):

                    H_interacting += w[idx, j, k, l] * (raise_L[idx] @ raise_L[j] @ lower_L[k] @ lower_L[l])

        for i in range(idx):
            H_interacting += w[i, idx, idx, idx] * (raise_L[i] @ raise_L[idx] @ lower_L[idx] @ lower_L[idx])
        
        for k in range(idx):
            for l in range(idx):
                H_interacting += v[idx, idx, k, l] * (raise_L[idx] @ raise_L[idx] @ lower_L[k] @ lower_L[l] + raise_L[k] @ raise_L[l] @ lower_L[idx] @ lower_L[idx])
        
        for j in range(idx):
            for l in range(idx):
                H_interacting += x[idx, j, idx, l] * (raise_L[idx] @ raise_L[j] @ lower_L[idx] @ lower_L[l])

        H_L = H_noninteracting + H_interacting


    def extend_right():

        nonlocal a, a_t, RIstring, H_R, lower_R, raise_R, I, Z

        new_raise_R = np.kron(a_t, RIstring)
        new_lower_R = np.kron(a, RIstring)

        idx = len(lower_R)

        lower_R = [np.kron(Z, op) for op in lower_R]
        raise_R = [np.kron(Z, op) for op in raise_R]

        lower_R.append(new_lower_R)
        raise_R.append(new_raise_R)

        RIstring = np.kron(I, RIstring)

        H_noninteracting = np.kron(I, H_R) + t[nso-1-idx, nso-1-idx] * new_raise_R @ new_lower_R
        H_interacting = np.zeros_like(H_noninteracting)

        for j in range(idx):
            H_interacting += t[nso-1-idx, nso-1-j] * (raise_R[idx] @ lower_R[j] + raise_R[j] @ lower_R[idx])

        for j in range(idx):
            for k in range(idx):
                for l in range(idx):
                    H_interacting += w[nso-1-idx,nso-1-j,nso-1-k,nso-1-l] * (raise_R[idx] @ raise_R[j] @ lower_R[k] @ lower_R[l])

        for i in range(idx):
            H_interacting += w[nso-1-i, nso-1-idx, nso-1-idx, nso-1-idx] * (raise_R[i] @ raise_R[idx] @ lower_R[idx] @ lower_R[idx])

        for k in range(idx):
            for l in range(idx):
                H_interacting += v[nso-1-idx,nso-1-idx,nso-1-k,nso-1-l] * (raise_R[idx] @ raise_R[idx] @ lower_R[k] @ lower_R[l] + raise_R[k] @ raise_R[l] @ lower_R[idx] @ lower_R[idx])

        for j in range(idx):
            for l in range(idx):
                H_interacting += x[nso-1-idx, nso-1-j, nso-1-idx, nso-1-l] * (raise_R[idx] @ raise_R[j] @ lower_R[idx] @ lower_R[l])

        H_R = H_noninteracting + H_interacting

    
    current_length = 2

    while current_length < mf.mol.nao:
        energy, wfc = superblock()
        decimate(wfc)
        extend_left()
        extend_right()
        current_length += 1
    
    energy, wfc = superblock()

    return energy + mf.energy_nuc()


    


In [38]:
from pyscf import fci

cisolver = pyscf.fci.FCI(mf)
cisolver.kernel()

(-1.1516725449612355,
 FCIvector([[ 9.92767007e-01,  5.13478149e-16,  5.67456324e-03,
             -1.82540822e-16],
            [-1.47352346e-15, -7.64960729e-02, -1.30917166e-16,
             -4.53286081e-02],
            [ 5.67456324e-03, -9.96106226e-17, -5.05447671e-02,
             -4.87296192e-17],
            [-2.69413620e-17, -4.53286081e-02,  5.09496161e-17,
             -4.28191520e-02]]))

In [42]:
# no sparse, no decimate

lattice_construction_dmrg_retry(mf)

t shape: (8, 8)
nso: 8
Initial left and right Hamiltonians constructed and operators added to the list.
Current lengths:
Left: 2
Right: 2
Energy:  -1.3797356566067607
Left shape: 4
Right shape: 4
Energy:  -1.450267865430048
Left shape: 8
Right shape: 8
Energy:  -1.7459184946955149


-1.0308141556144337

In [ ]:
# result with sparse, decimate

lattice_construction_dmrg_retry(mf)

t shape: (8, 8)
nso: 8
Initial left and right Hamiltonians constructed and operators added to the list.
Current lengths:
Left: 2
Right: 2
Energy:  -1.3797356566067616
Energy:  -1.622996232357946
Energy:  -2.2342392375065088


-1.5191348984254276

In [ ]:
# result for without sp.sparse but with decimate

lattice_construction_dmrg_retry(mf)

t shape: (8, 8)
nso: 8
Initial left and right Hamiltonians constructed and operators added to the list.
Current lengths:
Left: 2
Right: 2
Energy:  -1.379735656606761
Energy:  -1.6558521280855154
Energy:  -3.317749981833168


-2.602645642752087

In [15]:
Z = [[1,0],[0,-1]]
a = [[1,0],[0,0]]
a = np.array(a)
Z = np.array(Z)

np.kron(Z, a) @ np.kron(Z,a)


array([[1, 0, 0, 0],
       [0, 0, 0, 0],
       [0, 0, 1, 0],
       [0, 0, 0, 0]])

In [6]:
lattice_construction_dmrg(mf)


8
2
H_l, H_r shapes: (4, 4) (4, 4)
(16, 1)
[-1.37973566]
H_l, H_r shapes: (8, 8) (8, 8)
(64, 1)
[4.01437625]
original shape:  (8, 8) (8, 8)
final shape:  (8, 8) (8, 8)
H_l, H_r shapes: (16, 16) (16, 16)
(256, 1)
[9.63633005]
original shape:  (16, 16) (16, 16)
final shape:  (16, 16) (16, 16)
(1024, 1)


array([16.61823558])

In [ ]:
    # # H_superblock = make_superblock(H_l, H_r, Lzstring, LIstring, RIstring, lower_L, lower_R, raise_L, raise_R)

    # H_superblock_sparse = sp.sparse.csr_matrix(H_superblock)

    # E0, Psi = sp.sparse.linalg.eigsh(H_superblock_sparse, 1)


    # # print("SHAPE:",np.shape(lower_L))


    # # H_noninteracting = np.kron(H_l, np.eye(4)) + np.kron(np.eye(4), H_r)

    # # # now we need interaction term.   



    # # H_interacting = np.zeros_like(H_noninteracting)

    # # length_L = 2
    # # length_R = 2

    # # for i in range(length_R):
    # #     for j in range(length_L):
    # #         H_interacting += t[nso-i-1,j] * (np.kron(Lzstring, raise_R[i]) @ np.kron(lower_L[j], RIstring) + np.kron(Lzstring, raise_L[j]) @ np.kron(lower_R[i], RIstring))
    # #         for k in range(length_L):
    # #             for l in range(length_L):
    # #                 H_interacting += (g[nso-i-1,l,j,k] - g[j,l,nso-i-1,k]) * np.kron(Lzstring, raise_R[i]) @ np.kron(raise_L[j], RIstring) @ np.kron(lower_L[k], RIstring) @ np.kron(lower_L[l], RIstring)
    
    # # for i in range(length_L):
    # #     for j in range(length_R):
    # #         for k in range(length_R):
    # #             for l in range(length_R):
    # #                 H_interacting += (g[i, nso-l-1, nso-j-1, nso-k-1] - g[nso-j-1, nso-l-1, i, nso-k-1]) * np.kron(raise_L[i], RIstring) @ np.kron(Lzstring, raise_R[j]) @ np.kron(Lzstring, lower_R[k]) @ np.kron(Lzstring, lower_R[l])

    # # for i in range(length_R):
    # #     for j in range(length_R):
    # #         for k in range(length_L):
    # #             for l in range(length_L):
    # #                 H_interacting += (g[nso-i-1, l, nso-j-1, k]) * (np.kron(Lzstring, raise_R[i]) @ np.kron(Lzstring, raise_R[j]) @ np.kron(lower_L[k], RIstring) @ np.kron(lower_L[l], RIstring) + np.kron(raise_L[k], RIstring) @ np.kron(raise_L[l], RIstring) @ np.kron(Lzstring, lower_R[i]) @ np.kron(Lzstring, lower_R[j]))
    
    # # for i in range(length_R):
    # #     for k in range(length_R):
    # #         for j in range(length_L):
    # #             for l in range(length_L):
    # #                 H_interacting += x[nso-i-1, l, j, nso-k-1] * np.kron(Lzstring, raise_R[i]) @ np.kron(raise_L[j], RIstring) @ np.kron(Lzstring, lower_R[k]) @ np.kron(lower_L[l], RIstring)

    # # H_superblock = H_noninteracting + H_interacting

    # # H_superblock_sparse = sp.sparse.csr_matrix(H_superblock)

    # # E0, Psi = sp.sparse.linalg.eigsh(H_superblock_sparse, 1)

    # # print(np.shape(Psi))

    # # print(H_interacting)

    # Psi_reshaped = np.reshape(Psi, (2**length_L, 2**length_R))

    # dm_L = np.einsum('ij,kj->ik', Psi_reshaped, Psi_reshaped)
    # dm_R = np.einsum("ij,ik->jk", Psi_reshaped, Psi_reshaped)

    # # print(np.shape(dm_L))
    # # print(np.shape(dm_R))

    # schmidt_L, basis_L = np.linalg.eigh(-dm_L)
    # schmidt_R, basis_R = np.linalg.eigh(-dm_R)

    # schmidt_L = -schmidt_L
    # schmidt_R = -schmidt_R


    # if length_L > 4 and length_R > 4:
    #     mask_L = schmidt_L > 0.5
    #     mask_R = schmidt_R > 0.5
    #     basis_L = basis_L[:, mask_L]
    #     basis_R = basis_R[:, mask_R]

    # # print(basis_L)
    # # print(basis_L_filtered)
    # # print(schmidt_R)

    # # why does Garnet expand by two sites at a time rather than just one?

    # H_L_new = np.einsum('ij,ia,jb->ab', H_l, basis_L, basis_L)
    # H_l = np.kron(H_L_new, I)

    # H_R_new = np.einsum('ij,ia,jb->ab', H_r, basis_R, basis_R)
    # H_r = np.kron(I, H_R_new)

    # new_raise_L = [np.einsum('ij,ia,jb->ab', op, basis_L, basis_L) for op in raise_L]
    # new_lower_L = [np.einsum('ij,ia,jb->ab', op, basis_L, basis_L) for op in lower_L]
    # new_raise_R = [np.einsum('ij,ia,jb->ab', op, basis_R, basis_R) for op in raise_R]
    # new_lower_R = [np.einsum('ij,ia,jb->ab', op, basis_R, basis_R) for op in lower_R]

    # new_raise_L_I = [np.kron(op, I) for op in new_raise_L]
    # new_lower_L_I = [np.kron(op, I) for op in new_lower_L]
    # Z_new_raise_R = [np.kron(Z, op) for op in new_raise_R]
    # Z_new_lower_R = [np.kron(Z, op) for op in new_lower_L]

    # raise_L = new_raise_L_I
    # lower_L = new_lower_L_I
    # raise_R = Z_new_raise_R
    # lower_R = Z_new_lower_R

    # # we need to also construct new identity and Z operators

    # new_LIstring = np.einsum('ij,ia,jb->ab', LIstring, basis_L, basis_L)
    # new_Lzstring = np.einsum('ij,ia,jb->ab', Lzstring, basis_L, basis_L)
    # new_RIstring = np.einsum('ij,ia,jb->ab', RIstring, basis_R, basis_R)

In [86]:
matrix = np.eye(6) + np.diag([1, 0, 0, 0, 0, 0])

e, C = np.linalg.eigh(matrix)

np.shape(e)[0]

6

In [ ]:
# assume we begin from a unrestricted HF soln.
def L_annihil(i):
    if i == 0:
        op = [  
            [],
            []
        ]

def L_create(i):



def R_create(i, n):


def R_annihil(i, n):




SyntaxError: incomplete input (661871502.py, line 3)